# Imports

In [1]:
#Add notebooks directory to system path, otherwise we get a ModuleNotFoundError (utils is in different directory than execute)
import sys
import os

current_dir = os.getcwd()
notebooks_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))
if notebooks_dir not in sys.path:
    sys.path.append(notebooks_dir)

In [2]:
import pandas as pd
from notebooks.utils.util_datasources import Util_DataSources
u_datasources = Util_DataSources()

In [3]:
%load_ext autoreload
%autoreload 2

# Preprocess CDS data

In [4]:
#Define the path where the CSV files are located and load the sheets
path_cds = r'../../data/CDS_energy'
dataframes_cds = u_datasources.load_cds_excel_sheets(path_cds)

DataFrame for series: ElectricityDemand
DataFrame for series: GlobalHorizontalIrradiance
DataFrame for series: Hydropower(Reservoir)
DataFrame for series: Hydropower(RunOfRiver)
DataFrame for series: MeanSeaLevelPressure
DataFrame for series: SolarPVPower
DataFrame for series: AirTemperature
DataFrame for series: TotalPrecipitation
DataFrame for series: WindPowerOnshore
DataFrame for series: WindSpeed


In [5]:
#Define country mapping to match the names of the CDS countries (e.g. AT) with the Eurostat countries (e.g. Austria)
country_mapping = {
    'AT': 'Austria',                            
    'BE': 'Belgium',
    'BG': 'Bulgaria',
    'CY': 'Cyprus',
    'CZ': 'Czechia',
    'DE': 'Germany',
    'DK': 'Denmark',
    'EE': 'Estonia',
    'EL': 'Greece',
    'ES': 'Spain',
    'FI': 'Finland',
    'FR': 'France',
    'HR': 'Croatia',
    'HU': 'Hungary',
    'IE': 'Ireland',
    'IT': 'Italy',
    'LT': 'Lithuania',
    'LU': 'Luxembourg',
    'LV': 'Latvia',
    'MT': 'Malta',
    'NL': 'Netherlands',
    'PL': 'Poland',
    'PT': 'Portugal',
    'RO': 'Romania',
    'SE': 'Sweden',
    'SI': 'Slovenia',
    'SK': 'Slovakia'
}

In [6]:
dataframes_cds = u_datasources.prep_cds_data(dataframes_cds, country_mapping)
dataframes_cds["ElectricityDemand"].tail()


,Austria,Belgium,Bulgaria,Cyprus,Czechia,Germany,Denmark,Estonia,Greece,Spain,...,Netherlands,Poland,Portugal,Romania,Sweden,Slovenia,Slovakia,Date,month,year
545,5382919.5,6585216.0,2755298.4,468111.7,4664041.8,40768550.1,2516264.0,540553.9,4701797.7,19765830.1,...,8707732.9,11642191.3,3831525.0,4049300.0,9140570.2,1083855.1,2146917.6,2024-06-30,6,2024
546,5602134.7,6522406.8,2969592.9,525985.3,4589929.6,42497019.2,2453069.7,555848.9,5570632.6,22285185.8,...,9028432.8,12163587.3,4202602.9,4437472.8,8738464.9,1132328.6,2234681.2,2024-07-31,7,2024
547,5478978.0,6617350.0,2928635.7,492473.7,4595844.0,41979410.0,2627332.0,569205.8,4918699.1,21693921.6,...,9079795.9,12013452.2,3975031.0,4349262.0,9326540.2,1103395.9,2219671.8,2024-08-31,8,2024
548,5518915.8,6703801.5,2681389.1,380081.5,4719968.6,42391456.5,2631906.5,574359.8,3861294.9,19519859.0,...,8947579.2,11977192.1,3890237.5,4074973.9,9787318.9,1094479.9,2207443.9,2024-09-30,9,2024
549,5850426.2,7235275.7,2980063.0,320047.8,5188043.0,44818238.5,2850856.4,657711.8,3681263.4,19861477.9,...,9480666.2,12937504.6,4029799.3,4346240.9,11298867.3,1134794.0,2394137.9,2024-10-31,10,2024


# Preprocess eurostat data

In [7]:
#Define path and prepare data
file_path_eurostat = r'../../data/nrg_cb_em_spreadsheet.xlsx'
dataframes_eurostat = u_datasources.load_eurostat_excel_sheets(file_path_eurostat)
cleaned_dfs_eurostat = u_datasources.extract_eurostat_data(dataframes_eurostat)

DataFrame for sheet: Summary
DataFrame for sheet: Structure
DataFrame for sheet: Sheet 1
DataFrame for sheet: Sheet 2
DataFrame for sheet: Sheet 3
DataFrame for sheet: Sheet 4
DataFrame for sheet: Sheet 5
DataFrame for sheet: Sheet 6
DataFrame for sheet: Sheet 7


In [8]:
#Preprocess Eurostat data and make each Feature (e.g. Imports) a dataframe
dataframes_eurostat = {}
name_list = ["Imports", "Imports_from_EU", "Exports", "Exports_to_EU", "Transformation_Input", "Distribution_losses", "Available_to_internal_market"]

for i in range(2, len(cleaned_dfs_eurostat)):
    overview = cleaned_dfs_eurostat[i]
    overview = u_datasources.prep_eurostat_data(overview)
    columns_to_keep = list(country_mapping.values())
    columns_to_keep += ["Date", "month", "year"]
    overview = overview[columns_to_keep]
    dataframes_eurostat[name_list[i - 2]] = overview

c:\Users\linde\OneDrive\Dokumente\Master\Nowcasting Challenge\notebooks\utils\util_datasources.py:119: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.bfill()
c:\Users\linde\OneDrive\Dokumente\Master\Nowcasting Challenge\notebooks\utils\util_datasources.py:119: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.bfill()
c:\Users\linde\OneDrive\Dokumente\Master\Nowcasting Challenge\notebooks\utils\util_datasources.py:119: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future ver

In [9]:
dataframes_eurostat["Imports"].head()

TIME,Austria,Belgium,Bulgaria,Cyprus,Czechia,Germany,Denmark,Estonia,Greece,Spain,...,Netherlands,Poland,Portugal,Romania,Sweden,Slovenia,Slovakia,Date,month,year
0,2503.0,1588.0,467.0,0.0,1064.0,3525.0,753.0,17.0,130.0,775.0,...,2273.0,1008.0,894.0,90.0,840.0,587.0,1153.0,2008-01-31,1,2008
1,2283.0,1569.0,544.0,0.0,865.0,3182.0,862.0,10.0,215.0,613.0,...,1881.0,962.0,858.0,65.0,894.0,553.0,995.0,2008-02-29,2,2008
2,2162.0,2131.0,318.0,0.0,708.0,2999.0,897.0,221.0,419.0,230.0,...,2454.0,950.0,1079.0,60.0,933.0,674.0,724.0,2008-03-31,3,2008
3,1764.0,1886.0,141.0,0.0,569.0,3204.0,1366.0,164.0,327.0,257.0,...,2081.0,699.0,959.0,45.0,379.0,575.0,736.0,2008-04-30,4,2008
4,1287.0,1603.0,118.0,0.0,507.0,3469.0,1280.0,153.0,242.0,564.0,...,2376.0,583.0,854.0,58.0,351.0,456.0,431.0,2008-05-31,5,2008


# Concatenate Eurostat and CDS data

In [10]:
#List to hold DataFrames with the new 'type' column
dataframes_with_type = []

#Process dataframes_cds
for key, df in dataframes_cds.items():
    df['type'] = key  #Add the key as a new column
    dataframes_with_type.append(df)

#Process dataframes_eurostat
for key, df in dataframes_eurostat.items():
    df['type'] = key
    dataframes_with_type.append(df)

#Concatenate all DataFrames into one
result_df = pd.concat(dataframes_with_type, axis=0, ignore_index=True)

# Create dict with individual df for each country

In [11]:
#Create the final dict with country name as key and all data for this country (DataFrame) as value
country_dataframes = u_datasources.create_country_dict_with_starting_year(result_df,2008)

In [12]:
#Remove Malta since there are to little values
del country_dataframes['Malta']

#Write the country dict as the final form of data preparation in an excel file
with pd.ExcelWriter('countries_data.xlsx') as writer:
    for country, df in country_dataframes.items():
        df.to_excel(writer, sheet_name=country)